# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Document:** RFC 6749, *The OAuth 2.0 Authorization Framework* (IETF Standards Track), 76 pages.
**LLM:** Google Gemini API  |  **Bi-Encoder:** `sentence-transformers/all-MiniLM-L6-v2`  |  **Cross-Encoder:** `cross-encoder/ms-marco-MiniLM-L-6-v2`

Both pipelines run over the **exact same chunk corpus**:

| | Pipeline 1: Baseline | Pipeline 2: Re-Ranking |
|---|---|---|
| Stage 1 | Bi-encoder → FAISS → Top 10 | *same* Top 10 |
| Stage 2 | take Top 3 as-is | Cross-encoder scores all 10 (query, chunk) pairs → re-sort → Top 3 |
| Generation | Gemini → **Answer 1** | Gemini (identical prompt) → **Answer 2** |

Notebook outline:
1. Setup & configuration
2. Load the PDF and clean the RFC page layout
3. Section-aware chunking
4. Pipeline 1: bi-encoder index + baseline retrieval
5. Pipeline 2: cross-encoder re-ranking
6. LLM answer generation (Gemini)
7. Failure-mode analysis (stress tests)
8. Latency profiling
9. Scaling experiment (why not a cross-encoder over everything?)

## 1. Setup & Configuration

In [1]:
# Run once if the packages are missing:
# !pip install pymupdf sentence-transformers faiss-cpu google-genai pandas

In [2]:
import os
import re
import time
import statistics

import fitz                      # PyMuPDF: PDF text extraction
import numpy as np
import pandas as pd
import faiss                     # vector store (exact inner-product index)
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

In [3]:
# ---- Configuration ---------------------------------------------------------
# RFC 6749 rendered page-for-page (76 pages) from the official RFC Editor text:
#   https://www.rfc-editor.org/rfc/rfc6749.txt
PDF_PATH = "data/RFC6749_OAuth2.pdf"

BI_ENCODER_NAME    = "sentence-transformers/all-MiniLM-L6-v2"   # dense bi-encoder
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # re-ranker
GEMINI_MODEL       = "gemini-3.1-flash-lite"                    # other options: "gemini-3.8-flash" (often overloaded on the free tier)

TOP_K = 10          # candidates retrieved by the bi-encoder
TOP_N = 3           # chunks passed to the LLM
MAX_CHUNK_WORDS = 180

# The Gemini key is read from the environment so it never ends up in the notebook / GitHub.
#   Linux/macOS:  export GEMINI_API_KEY="your-key"   (then start Jupyter from that terminal)
# Get a free key at https://aistudio.google.com/apikey
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

## 2. Load the PDF and Clean the RFC Page Layout

RFC 6749 is a 76-page IETF specification. Extracting its text naively would mix in content that is not part of the rules:

* a **running header** (`RFC 6749   OAuth 2.0   October 2012`) and **footer** (`Hardt   Standards Track   [Page N]`) on every page,
* **front matter**: title page, abstract, copyright notice and a 3-page **table of contents** (which repeats every section title and would attract queries),
* **back matter**: IANA registrations (Section 11), references (Section 12) and appendices,
* **ASCII-art diagrams** (protocol flow boxes and arrows) that carry almost no words.

We drop the header/footer lines with regular expressions, start at the real `1.  Introduction` heading (ToC entries are recognised by
their dotted leaders), stop before Section 11, and remove diagram lines: lines where fewer than 40% of the characters are letters or digits, or that contain box borders (`+--`), arrows (`-->`) or vertical bars (`|`).

In [4]:
PAGE_HEADER = re.compile(r"^RFC 6749\s+OAuth 2\.0\s+October 2012$")
PAGE_FOOTER = re.compile(r"^Hardt\s+Standards Track\s+\[Page \d+\]$")
HEADING     = re.compile(r"^(\d+(?:\.\d+)*)\.\s+(\S.*)$")      # "4.1.2.1.  Error Response" at column 0
FIRST_SECTION, STOP_SECTIONS = "1", ("11", "12")                # body = sections 1-10; drop IANA (11) + References (12)


DIAGRAM_MARK = re.compile(r"\+-{2,}|-{2,}>|<-{2,}|\|\s|\s\|")   # box borders, arrows, vertical bars


def is_diagram(line):
    # ASCII-art lines (boxes, arrows, labels inside boxes): few letters/digits, or drawing characters
    s = line.strip()
    return bool(s) and (sum(ch.isalnum() for ch in s) / len(s) < 0.4 or bool(DIAGRAM_MARK.search(s)))


def extract_lines(pdf_path):
    # Return the body lines of the RFC with page furniture, front matter and back matter removed.
    doc = fitz.open(pdf_path)
    lines = []
    for page in doc:
        for line in page.get_text().split("\n"):
            if PAGE_HEADER.match(line.strip()) or PAGE_FOOTER.match(line.strip()):
                continue                               # running header / footer on every page
            lines.append(line.rstrip())

    # Skip the title page, abstract and table of contents: the body starts at the
    # "1.  Introduction" heading that is NOT a ToC entry (ToC entries end with dotted leaders + page no.)
    start = next(i for i, l in enumerate(lines) if HEADING.match(l) and HEADING.match(l).group(1) == FIRST_SECTION and "...." not in l)
    body = []
    for l in lines[start:]:
        m = HEADING.match(l)
        if m and m.group(1).split(".")[0] in STOP_SECTIONS:
            break                                      # IANA considerations + references: not rules
        if l.startswith("Appendix A"):
            break
        body.append(l)
    return body


body_lines = extract_lines(PDF_PATH)
diagram_lines = sum(is_diagram(l) for l in body_lines)
print(f"{len(body_lines)} body lines kept ({diagram_lines} ASCII-diagram lines will be dropped)")
print("\n".join(body_lines[:12]))

2104 body lines kept (124 ASCII-diagram lines will be dropped)
1.  Introduction
   In the traditional client-server authentication model, the client
   requests an access-restricted resource (protected resource) on the
   server by authenticating with the server using the resource owner's
   credentials.  In order to provide third-party applications access to
   restricted resources, the resource owner shares its credentials with
   the third party.  This creates several problems and limitations:
   o  Third-party applications are required to store the resource
      owner's credentials for future use, typically a password in
      clear-text.
   o  Servers are required to support password authentication, despite
      the security weaknesses inherent in passwords.


## 3. Section-Aware Chunking

An RFC is organised into **numbered sections** (`4.1.2.1.  Error Response`), and each normative rule (MUST / MUST NOT / SHOULD NOT) belongs
to one of them. Instead of cutting the text every *N* characters (which would split a rule from its exception), we:

1. Detect each section heading at the start of a line (`4.4.3.  Access Token Response`) and collect the paragraphs under it.
2. Drop headings with no text of their own (e.g. `4.  Obtaining Authorization`, whose content lives in its subsections).
3. Pack each section's paragraphs into chunks of at most `MAX_CHUNK_WORDS` words, so long sections become several chunks.

Every chunk is prefixed with `Section N (Title):` so the LLM can cite it. The **same chunk list** is used by both pipelines.

In [5]:
def split_sections(lines):
    # Group lines under their numbered heading; paragraphs are separated by blank lines.
    sections, current, para = [], None, []
    def flush_para():
        if para and current is not None:
            current["paras"].append(" ".join(para))
        para.clear()
    for l in lines:
        m = HEADING.match(l)
        if m:
            flush_para()
            current = {"section": m.group(1), "title": m.group(2).strip(), "paras": []}
            sections.append(current)
        elif not l.strip():
            flush_para()
        elif not is_diagram(l):
            para.append(" ".join(l.split()))
    flush_para()
    return [s for s in sections if s["paras"]]        # headings with no own text (e.g. "4.  Obtaining Authorization") are dropped


def pack_paragraphs(paras, max_words):
    # Greedily pack consecutive paragraphs into chunks of <= max_words (splitting oversized ones).
    chunks, buf, n = [], [], 0
    for text in paras:
        words = text.split()
        while len(words) > max_words:
            if buf:
                chunks.append(" ".join(buf)); buf, n = [], 0
            chunks.append(" ".join(words[:max_words]))
            words = words[max_words:]
        if n + len(words) > max_words and buf:
            chunks.append(" ".join(buf)); buf, n = [], 0
        buf.append(" ".join(words)); n += len(words)
    if buf:
        chunks.append(" ".join(buf))
    return chunks


sections = split_sections(body_lines)
chunks = []
for s in sections:
    for part in pack_paragraphs(s["paras"], MAX_CHUNK_WORDS):
        chunks.append({"chunk_id": len(chunks), "section": s["section"], "title": s["title"],
                       "text": f"Section {s['section']} ({s['title']}): {part}"})

chunk_df = pd.DataFrame(chunks)
print(f"{len(sections)} sections -> {len(chunks)} chunks")
print("Words per chunk: min %d / mean %.0f / max %d" % tuple(
    f(chunk_df.text.str.split().str.len()) for f in (np.min, np.mean, np.max)))
chunk_df[["chunk_id", "section", "title"]].head(12)

83 sections -> 132 chunks
Words per chunk: min 7 / mean 115 / max 192


,chunk_id,section,title
0,0,1,Introduction
1,1,1,Introduction
2,2,1,Introduction
3,3,1.1,Roles
4,4,1.2,Protocol Flow
5,5,1.2,Protocol Flow
6,6,1.3,Authorization Grant
7,7,1.3.1,Authorization Code
8,8,1.3.2,Implicit
9,9,1.3.2,Implicit


In [6]:
# Quick sanity check: the clauses the stress tests depend on must be present and intact
for needle in ["A refresh token SHOULD NOT be included", "MUST NOT issue a refresh token",
               "MUST NOT include any scope not originally granted", "MUST be kept confidential in transit"]:
    hits = chunk_df[chunk_df.text.str.contains(needle)]
    print(f"{needle!r:55} -> chunks {hits.chunk_id.tolist()} (section {hits.section.tolist()})")

'A refresh token SHOULD NOT be included'                -> chunks [84] (section ['4.4.3'])
'MUST NOT issue a refresh token'                        -> chunks [70] (section ['4.2.2'])
'MUST NOT include any scope not originally granted'     -> chunks [94] (section ['6'])
'MUST be kept confidential in transit'                  -> chunks [112, 114] (section ['10.3', '10.4'])


## 4. Pipeline 1: Bi-Encoder Index + Baseline Retrieval

A **bi-encoder** turns the query and each chunk into vectors *independently*. Chunk vectors are computed once, offline, and stored in a vector index.
At query time only the query is encoded, then a nearest-neighbour search finds the closest chunks.

Embeddings are L2-normalised, so the **inner product in `faiss.IndexFlatIP` equals cosine similarity**.

In [7]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME)

# Offline indexing: embed every chunk once
chunk_texts = chunk_df.text.tolist()
chunk_embeddings = bi_encoder.encode(chunk_texts, normalize_embeddings=True, convert_to_numpy=True).astype("float32")

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)
print(f"FAISS index: {index.ntotal} vectors x {chunk_embeddings.shape[1]} dims")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index: 132 vectors x 384 dims


In [8]:
def baseline_retrieve(query, k=TOP_K):
    # Stage 1: encode the query and return the top-k chunks by cosine similarity.
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q, k)
    return [{"chunk_id": int(i), "bi_rank": r, "bi_score": float(s)}
            for r, (i, s) in enumerate(zip(ids[0], scores[0]), start=1)]


baseline_retrieve("Must the client authenticate at the token endpoint?")[:5]

[{'chunk_id': 122, 'bi_rank': 1, 'bi_score': 0.6548409461975098},
 {'chunk_id': 45, 'bi_rank': 2, 'bi_score': 0.6493399143218994},
 {'chunk_id': 79, 'bi_rank': 3, 'bi_score': 0.6478606462478638},
 {'chunk_id': 43, 'bi_rank': 4, 'bi_score': 0.6203311681747437},
 {'chunk_id': 78, 'bi_rank': 5, 'bi_score': 0.6174899339675903}]

## 5. Pipeline 2: Cross-Encoder Re-Ranking

A **cross-encoder** reads the query and a chunk *together* as one input (`[CLS] query [SEP] chunk [SEP]`), so every query token can attend to every
chunk token. It outputs a single relevance score. This is much more precise, but nothing can be precomputed: it has to run once per (query, chunk) pair.
So we only apply it to the 10 candidates from Stage 1.

In [9]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)


def rerank(query, candidates):
    # Stage 2: score every (query, chunk) pair jointly and re-sort descending by cross-encoder score.
    pairs = [(query, chunk_texts[c["chunk_id"]]) for c in candidates]
    ce_scores = cross_encoder.predict(pairs)
    reranked = [dict(c, ce_score=float(s)) for c, s in zip(candidates, ce_scores)]
    reranked.sort(key=lambda c: c["ce_score"], reverse=True)
    for r, c in enumerate(reranked, start=1):
        c["ce_rank"] = r
    return reranked


rerank("Must the client authenticate at the token endpoint?",
       baseline_retrieve("Must the client authenticate at the token endpoint?"))[:3]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[{'chunk_id': 122,
  'bi_rank': 1,
  'bi_score': 0.6548409461975098,
  'ce_score': 8.189594268798828,
  'ce_rank': 1},
 {'chunk_id': 83,
  'bi_rank': 8,
  'bi_score': 0.6026461124420166,
  'ce_score': 7.912633895874023,
  'ce_rank': 2},
 {'chunk_id': 45,
  'bi_rank': 2,
  'bi_score': 0.6493399143218994,
  'ce_score': 7.833852767944336,
  'ce_rank': 3}]

## 6. LLM Answer Generation (Gemini)

Both pipelines use the **identical prompt template**. The only difference is which 3 chunks fill the context.
The prompt tells the model to answer only from the context, so any wrong answer can be traced back to retrieval.
Rate-limit (429) and overload (503) errors are retried with a growing wait.
`temperature=0` keeps the answers as deterministic as possible, so Answer 1 vs Answer 2 differences come from the context, not from sampling.

In [10]:
PROMPT_TEMPLATE = '''You are an expert assistant answering questions about the OAuth 2.0 Authorization Framework (RFC 6749).
Answer the question using ONLY the context below. Cite the section numbers you rely on.
If the context does not contain the answer, say "The provided context does not answer this."

Context:
{context}

Question: {question}

Answer (concise, 2-5 sentences):'''

client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None
if client is None:
    print("GEMINI_API_KEY not set: retrieval will run, LLM answers will be skipped.")


def build_prompt(question, chunk_ids):
    context = "\n\n".join(f"[{i + 1}] {chunk_texts[c]}" for i, c in enumerate(chunk_ids))
    return PROMPT_TEMPLATE.format(context=context, question=question)


RETRYABLE = ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE")   # rate limit / model overloaded


def generate_answer(question, chunk_ids, retries=6):
    if client is None:
        return "[skipped: GEMINI_API_KEY not set]"
    prompt = build_prompt(question, chunk_ids)
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0),
            )
            return response.text.strip()
        except Exception as e:                      # free tier: back off on rate limits / overload, then retry
            if any(code in str(e) for code in RETRYABLE) and attempt < retries - 1:
                time.sleep(20 * (attempt + 1))
            else:
                raise

In [11]:
# Optional: list the Gemini models your key can use (handy if GEMINI_MODEL has been retired)
# if client: print([m.name for m in client.models.list() if "flash" in m.name])

In [12]:
def run_pipelines(question, generate=True):
    # Run Pipeline 1 and Pipeline 2 for one question and return everything needed for analysis.
    t0 = time.perf_counter()
    candidates = baseline_retrieve(question)
    t1 = time.perf_counter()
    reranked = rerank(question, candidates)
    t2 = time.perf_counter()

    baseline_top = [c["chunk_id"] for c in candidates[:TOP_N]]
    reranked_top = [c["chunk_id"] for c in reranked[:TOP_N]]
    return {
        "question": question,
        "candidates": reranked,                     # each has bi_rank/bi_score and ce_rank/ce_score
        "baseline_top": baseline_top,
        "reranked_top": reranked_top,
        "answer_1": generate_answer(question, baseline_top) if generate else None,
        "answer_2": generate_answer(question, reranked_top) if generate else None,
        "t_baseline_ms": (t1 - t0) * 1000,
        "t_rerank_ms":   (t2 - t1) * 1000,
    }


def rank_table(result, gold_sections=()):
    # Rank-shift table: bi-encoder rank vs cross-encoder rank for all 10 candidates.
    rows = []
    for c in sorted(result["candidates"], key=lambda c: c["bi_rank"]):
        ch = chunks[c["chunk_id"]]
        rows.append({
            "gold": "★" if ch["section"] in gold_sections else "",
            "chunk": c["chunk_id"],
            "section": ch["section"],
            "title": ch["title"][:45],
            "bi_rank": c["bi_rank"],
            "bi_score": round(c["bi_score"], 3),
            "ce_rank": c["ce_rank"],
            "ce_score": round(c["ce_score"], 2),
            "shift": f"#{c['bi_rank']} → #{c['ce_rank']}",
        })
    return pd.DataFrame(rows)

In [13]:
# Smoke test on an easy question: both pipelines should agree here
demo = run_pipelines("What is an access token?")
print("Baseline top-3:", demo["baseline_top"], "| Re-ranked top-3:", demo["reranked_top"])
print("\nAnswer 1:", demo["answer_1"])
print("\nAnswer 2:", demo["answer_2"])

Baseline top-3: [12, 113, 13] | Re-ranked top-3: [12, 13, 96]

Answer 1: An access token is a string representing an authorization issued to the client that is used to access protected resources [1]. It acts as an abstraction layer, replacing various authorization constructs with a single token understood by the resource server [1]. The token may be opaque to the client, represent specific scopes and durations of access, and either serve as an identifier to retrieve authorization information or self-contain that information in a verifiable manner [1].

Answer 2: An access token is a string representing an authorization granted by the resource owner and issued to the client, which is used to access protected resources [1]. It serves as an abstraction layer that replaces various authorization constructs with a single token understood by the resource server [1]. The token may be opaque to the client, or it may self-contain authorization information in a verifiable manner [1].


## 7. Failure-Mode Analysis (Stress Testing)

Each test query has a **gold section**: the provision that actually answers it (found by reading the Act).
For each query we show:
* the **rank-shift table** for all 10 candidates (★ = gold chunk),
* whether the gold chunk was in the **top 3 before / after** re-ranking,
* **Answer 1** (baseline context) vs **Answer 2** (re-ranked context).

In [14]:
# category, question, gold section(s), what the correct answer hinges on
STRESS_TESTS = [{'category': 'Keyword Trap / Distractor',
  'question': 'Must access tokens be kept confidential in transit and storage?',
  'gold': ['10.3'],
  'why': "s.10.3 (Security Considerations): access token credentials MUST be kept confidential in transit and storage. The words 'access tokens' "
         'pull up the general definition in s.1.4, which says nothing about confidentiality.'},
 {'category': 'Exceptions & Corner Cases',
  'question': 'In the client credentials flow, should the access token response contain a refresh token?',
  'gold': ['4.4.3'],
  'why': 'General rule (s.1.5, s.5.1): the server may issue a refresh token with the access token. Exception in s.4.4.3: for the client credentials '
         "grant 'A refresh token SHOULD NOT be included'."},
 {'category': 'Keyword Trap (synonym / role)',
  'question': 'Does the token response for a client acting on its own behalf include a refresh token?',
  'gold': ['4.4.3'],
  'why': "A client acting on its own behalf (not for a user) is the client credentials grant (s.4.4). The spec never uses the query's wording, and "
         's.4.4.3 says a refresh token SHOULD NOT be included.'},
 {'category': 'Negation & Constraints',
  'question': 'Can a refreshed access token have a broader scope than the original one?',
  'gold': ['6'],
  'why': "s.6: the requested scope MUST NOT include any scope not originally granted by the resource owner. The general refresh-token chunks (s.1.5) only say 'identical or narrower scope'; the actual prohibition is in s.6."},
 {'category': 'Other: Buried chunk',
  'question': 'Which parameters are required in the authorization request?',
  'gold': ['4.1.1'],
  'why': 's.4.1.1 lists response_type and client_id as REQUIRED, redirect_uri and scope as OPTIONAL and state as RECOMMENDED. General endpoint text '
         '(s.3.1) and the implicit-grant request (s.4.2.1) rank above it.'},
 {'category': 'Limitation: re-ranking demotes the answer',
  'question': 'Must the token endpoint response include Cache-Control no-store?',
  'gold': ['5.1'],
  'why': "s.5.1: the server MUST include 'Cache-Control: no-store' and 'Pragma: no-cache'. The bi-encoder finds it at #1, but the cross-encoder "
         'prefers grant-specific response sections and pushes it out of the top 3.'},
 {'category': 'Control: rule stated in two places',
  'question': 'In the implicit flow, can the access token response include a refresh token?',
  'gold': ['4.2', '4.2.2'],
  'why': "The answer appears twice: s.4.2 says the implicit grant 'does not support the issuance of refresh tokens' and s.4.2.2 says 'The authorization server MUST NOT issue a refresh token.' Because s.4.2 is retrieved by both pipelines, both should answer correctly."}]

In [15]:
def show_case(test):
    result = run_pipelines(test["question"])
    table = rank_table(result, test["gold"])
    gold_rows = table[table.gold == "★"]
    before = gold_rows.bi_rank.min() if len(gold_rows) else None
    after = gold_rows.ce_rank.min() if len(gold_rows) else None

    print("=" * 110)
    print(f"[{test['category']}] {test['question']}")
    print(f"Gold: Section(s) {', '.join(test['gold'])}: {test['why']}")
    print(f"Best gold rank: baseline #{before} → re-ranked #{after}")
    display(table)
    print("Answer 1 (baseline top-3", result["baseline_top"], "):\n", result["answer_1"], "\n")
    print("Answer 2 (re-ranked top-3", result["reranked_top"], "):\n", result["answer_2"])
    return {"category": test["category"], "question": test["question"],
            "gold_rank_baseline": before, "gold_rank_reranked": after,
            "gold_in_top3_baseline": before is not None and before <= TOP_N,
            "gold_in_top3_reranked": after is not None and after <= TOP_N,
            "answer_1": result["answer_1"], "answer_2": result["answer_2"]}

In [16]:
case_1 = show_case(STRESS_TESTS[0])

[Keyword Trap / Distractor] Must access tokens be kept confidential in transit and storage?
Gold: Section(s) 10.3: s.10.3 (Security Considerations): access token credentials MUST be kept confidential in transit and storage. The words 'access tokens' pull up the general definition in s.1.4, which says nothing about confidentiality.
Best gold rank: baseline #4 → re-ranked #1


,gold,chunk,section,title,bi_rank,bi_score,ce_rank,ce_score,shift
0,,13,1.4,Access Token,1,0.612,8,-3.10,#1 → #8
1,,121,10.8,Request Confidentiality,2,0.596,2,4.77,#2 → #2
2,,12,1.4,Access Token,3,0.586,5,-0.14,#3 → #5
3,★,112,10.3,Access Tokens,4,0.585,1,9.68,#4 → #1
4,★,113,10.3,Access Tokens,5,0.535,7,-2.71,#5 → #7
5,,1,1,Introduction,6,0.505,10,-3.47,#6 → #10
6,,25,2.1,Client Types,7,0.498,9,-3.29,#7 → #9
7,,81,4.4,Client Credentials Grant,8,0.494,4,1.03,#8 → #4
8,,96,7,Accessing Protected Resources,9,0.483,6,-0.79,#9 → #6
9,,79,4.3.2,Access Token Request,10,0.480,3,1.08,#10 → #3


Answer 1 (baseline top-3 [13, 121, 12] ):
 Yes, access tokens must not be transmitted in the clear, meaning they must be kept confidential in transit. This requirement is specified in Section 10.8. 

Answer 2 (re-ranked top-3 [112, 121, 79] ):
 Yes, access token credentials and any confidential access token attributes must be kept confidential in both transit and storage. They must only be shared among the authorization server, the resource servers for which the token is valid, and the client to whom the token was issued (Section 10.3). Furthermore, access tokens must not be transmitted in the clear and must be transmitted using TLS (Section 10.3, Section 10.8).


In [17]:
case_2 = show_case(STRESS_TESTS[1])

[Exceptions & Corner Cases] In the client credentials flow, should the access token response contain a refresh token?
Gold: Section(s) 4.4.3: General rule (s.1.5, s.5.1): the server may issue a refresh token with the access token. Exception in s.4.4.3: for the client credentials grant 'A refresh token SHOULD NOT be included'.
Best gold rank: baseline #8 → re-ranked #1


,gold,chunk,section,title,bi_rank,bi_score,ce_rank,ce_score,shift
0,,14,1.5,Refresh Token,1,0.762,3,5.49,#1 → #3
1,,15,1.5,Refresh Token,2,0.762,4,5.19,#2 → #4
2,,94,6,Refreshing an Access Token,3,0.755,2,5.51,#3 → #2
3,,16,1.5,Refresh Token,4,0.751,7,4.11,#4 → #7
4,,95,6,Refreshing an Access Token,5,0.726,8,4.09,#5 → #8
5,,44,3.2.1,Client Authentication,6,0.705,10,2.46,#6 → #10
6,,114,10.4,Refresh Tokens,7,0.701,5,5.00,#7 → #5
7,★,84,4.4.3,Access Token Response,8,0.698,1,6.85,#8 → #1
8,,115,10.4,Refresh Tokens,9,0.698,9,2.75,#9 → #9
9,,88,5.1,Successful Response,10,0.691,6,4.90,#10 → #6


Answer 1 (baseline top-3 [14, 15, 94] ):
 The provided context does not answer this. While it notes that issuing a refresh token is optional at the discretion of the authorization server [1], it does not specify the requirements or behavior regarding refresh tokens within the client credentials flow. 

Answer 2 (re-ranked top-3 [84, 94, 14] ):
 In the client credentials flow, a refresh token should not be included in the access token response. According to Section 4.4.3, if the access token request is valid and authorized, the authorization server issues an access token but a refresh token SHOULD NOT be included.


In [18]:
case_3 = show_case(STRESS_TESTS[2])

[Keyword Trap (synonym / role)] Does the token response for a client acting on its own behalf include a refresh token?
Gold: Section(s) 4.4.3: A client acting on its own behalf (not for a user) is the client credentials grant (s.4.4). The spec never uses the query's wording, and s.4.4.3 says a refresh token SHOULD NOT be included.
Best gold rank: baseline #9 → re-ranked #1


,gold,chunk,section,title,bi_rank,bi_score,ce_rank,ce_score,shift
0,,14,1.5,Refresh Token,1,0.711,3,4.44,#1 → #3
1,,115,10.4,Refresh Tokens,2,0.698,4,4.11,#2 → #4
2,,16,1.5,Refresh Token,3,0.677,10,2.36,#3 → #10
3,,114,10.4,Refresh Tokens,4,0.666,2,5.08,#4 → #2
4,,94,6,Refreshing an Access Token,5,0.648,6,3.57,#5 → #6
5,,15,1.5,Refresh Token,6,0.639,9,2.42,#6 → #9
6,,88,5.1,Successful Response,7,0.623,5,3.80,#7 → #5
7,,95,6,Refreshing an Access Token,8,0.591,8,2.70,#8 → #8
8,★,84,4.4.3,Access Token Response,9,0.581,1,5.09,#9 → #1
9,,86,5,Issuing an Access Token,10,0.568,7,3.32,#10 → #7


Answer 1 (baseline top-3 [14, 115, 16] ):
 The provided context does not answer this. While it notes that issuing a refresh token is optional at the discretion of the authorization server [1], it does not specify whether a refresh token is included in the response for a client acting on its own behalf. 

Answer 2 (re-ranked top-3 [84, 114, 14] ):
 According to Section 4.4.3, when a client acts on its own behalf (the client credentials grant), a refresh token should not be included in the access token response. Section 1.5 further notes that the issuance of a refresh token is optional at the discretion of the authorization server.


In [19]:
case_4 = show_case(STRESS_TESTS[3])

[Negation & Constraints] Can a refreshed access token have a broader scope than the original one?
Gold: Section(s) 6: s.6: the requested scope MUST NOT include any scope not originally granted by the resource owner. The general refresh-token chunks (s.1.5) only say 'identical or narrower scope'; the actual prohibition is in s.6.
Best gold rank: baseline #3 → re-ranked #2


,gold,chunk,section,title,bi_rank,bi_score,ce_rank,ce_score,shift
0,,16,1.5,Refresh Token,1,0.731,4,1.02,#1 → #4
1,,14,1.5,Refresh Token,2,0.708,1,5.57,#2 → #1
2,★,94,6,Refreshing an Access Token,3,0.684,2,3.52,#3 → #2
3,,15,1.5,Refresh Token,4,0.683,6,0.04,#4 → #6
4,,115,10.4,Refresh Tokens,5,0.657,10,-1.84,#5 → #10
5,★,95,6,Refreshing an Access Token,6,0.625,5,0.73,#6 → #5
6,,88,5.1,Successful Response,7,0.604,3,3.01,#7 → #3
7,,84,4.4.3,Access Token Response,8,0.592,7,-0.86,#8 → #7
8,,86,5,Issuing an Access Token,9,0.589,8,-1.04,#9 → #8
9,,114,10.4,Refresh Tokens,10,0.585,9,-1.08,#10 → #9


Answer 1 (baseline top-3 [16, 14, 94] ):
 No, a refreshed access token cannot have a broader scope than the original one. According to Section 6, the requested scope in a refresh request must not include any scope not originally granted by the resource owner. If the scope parameter is omitted, it is treated as equal to the scope originally granted. 

Answer 2 (re-ranked top-3 [14, 94, 88] ):
 No, a refreshed access token cannot have a broader scope than the original one. According to Section 6, the requested scope in a refresh request must not include any scope not originally granted by the resource owner. Additionally, Section 1.5 notes that refresh tokens are used to obtain new access tokens with identical or narrower scope.


In [20]:
case_5 = show_case(STRESS_TESTS[4])

[Other: Buried chunk] Which parameters are required in the authorization request?
Gold: Section(s) 4.1.1: s.4.1.1 lists response_type and client_id as REQUIRED, redirect_uri and scope as OPTIONAL and state as RECOMMENDED. General endpoint text (s.3.1) and the implicit-grant request (s.4.2.1) rank above it.
Best gold rank: baseline #5 → re-ranked #3


,gold,chunk,section,title,bi_rank,bi_score,ce_rank,ce_score,shift
0,,34,3.1,Authorization Endpoint,1,0.752,4,6.89,#1 → #4
1,,67,4.2.1,Authorization Request,2,0.692,1,7.22,#2 → #1
2,,100,8.2,Defining New Endpoint Parameters,3,0.686,6,6.27,#3 → #6
3,,33,3.1,Authorization Endpoint,4,0.661,10,1.40,#4 → #10
4,★,52,4.1.1,Authorization Request,5,0.659,3,6.94,#5 → #3
5,,78,4.3.2,Access Token Request,6,0.654,7,4.96,#6 → #7
6,,82,4.4.1,Authorization Request and Response,7,0.653,9,1.42,#7 → #9
7,,60,4.1.3,Access Token Request,8,0.644,2,7.01,#8 → #2
8,,35,3.1.1,Response Type,9,0.643,5,6.77,#9 → #5
9,,29,2.3.1,Client Password,10,0.643,8,2.85,#10 → #8


Answer 1 (baseline top-3 [34, 67, 100] ):
 The required parameters for the authorization request are `response_type` (which must be set to "token") and `client_id` (the client identifier). These are specified in Section 4.2.1. 

Answer 2 (re-ranked top-3 [67, 60, 52] ):
 The required parameters for an authorization request depend on the grant type. For the authorization code grant, the required parameters are `response_type` (set to "code") and `client_id` (Section 4.1.1). For the implicit grant, the required parameters are `response_type` (set to "token") and `client_id` (Section 4.2.1).


In [21]:
case_6 = show_case(STRESS_TESTS[5])

[Limitation: re-ranking demotes the answer] Must the token endpoint response include Cache-Control no-store?
Gold: Section(s) 5.1: s.5.1: the server MUST include 'Cache-Control: no-store' and 'Pragma: no-cache'. The bi-encoder finds it at #1, but the cross-encoder prefers grant-specific response sections and pushes it out of the top 3.
Best gold rank: baseline #1 → re-ranked #4


,gold,chunk,section,title,bi_rank,bi_score,ce_rank,ce_score,shift
0,★,88,5.1,Successful Response,1,0.521,4,3.99,#1 → #4
1,,62,4.1.4,Access Token Response,2,0.499,3,4.56,#2 → #3
2,,80,4.3.3,Access Token Response,3,0.492,2,4.56,#3 → #2
3,,42,3.2,Token Endpoint,4,0.489,7,-2.96,#4 → #7
4,,43,3.2,Token Endpoint,5,0.485,5,2.06,#5 → #5
5,,84,4.4.3,Access Token Response,6,0.472,1,5.83,#6 → #1
6,,93,5.2,Error Response,7,0.458,6,1.91,#7 → #6
7,★,87,5.1,Successful Response,8,0.426,10,-6.73,#8 → #10
8,,71,4.2.2,Access Token Response,9,0.418,8,-4.26,#9 → #8
9,,33,3.1,Authorization Endpoint,10,0.417,9,-5.88,#10 → #9


Answer 1 (baseline top-3 [88, 62, 80] ):
 Yes, the authorization server must include the HTTP "Cache-Control" response header field with a value of "no-store" in any response containing tokens, credentials, or other sensitive information [1]. 

Answer 2 (re-ranked top-3 [84, 80, 62] ):
 The provided context does not explicitly state that the response *must* include "Cache-Control: no-store". However, the example successful responses provided in sections 4.1.4, 4.3.3, and 4.4.3 all include "Cache-Control: no-store" as part of the HTTP response headers.


In [22]:
case_7 = show_case(STRESS_TESTS[6])

[Control: rule stated in two places] In the implicit flow, can the access token response include a refresh token?
Gold: Section(s) 4.2, 4.2.2: The answer appears twice: s.4.2 says the implicit grant 'does not support the issuance of refresh tokens' and s.4.2.2 says 'The authorization server MUST NOT issue a refresh token.' Because s.4.2 is retrieved by both pipelines, both should answer correctly.
Best gold rank: baseline #3 → re-ranked #2


,gold,chunk,section,title,bi_rank,bi_score,ce_rank,ce_score,shift
0,,15,1.5,Refresh Token,1,0.657,4,4.42,#1 → #4
1,,16,1.5,Refresh Token,2,0.649,9,1.83,#2 → #9
2,★,63,4.2,Implicit Grant,3,0.646,2,5.80,#3 → #2
3,,14,1.5,Refresh Token,4,0.636,6,3.43,#4 → #6
4,,94,6,Refreshing an Access Token,5,0.624,7,3.20,#5 → #7
5,,84,4.4.3,Access Token Response,6,0.610,1,5.93,#6 → #1
6,,130,10.16,Misuse of Access Token to Impersonate Resourc,7,0.597,5,4.36,#7 → #5
7,,80,4.3.3,Access Token Response,8,0.591,3,4.59,#8 → #3
8,,115,10.4,Refresh Tokens,9,0.591,10,0.63,#9 → #10
9,,95,6,Refreshing an Access Token,10,0.588,8,2.34,#10 → #8


Answer 1 (baseline top-3 [15, 16, 63] ):
 No, the implicit grant type does not support the issuance of refresh tokens. This is stated in Section 4.2. 

Answer 2 (re-ranked top-3 [84, 63, 80] ):
 No, the implicit grant type does not support the issuance of refresh tokens. This is explicitly stated in Section 4.2.


In [23]:
# Summary across all stress tests
summary = pd.DataFrame([case_1, case_2, case_3, case_4, case_5, case_6, case_7])
os.makedirs("results", exist_ok=True)
summary.to_csv("results/stress_test_summary.csv", index=False)
summary[["category", "question", "gold_rank_baseline", "gold_rank_reranked", "gold_in_top3_baseline", "gold_in_top3_reranked"]]

,category,question,gold_rank_baseline,gold_rank_reranked,gold_in_top3_baseline,gold_in_top3_reranked
0,Keyword Trap / Distractor,Must access tokens be kept confidential in transit and storage?,4,1,False,True
1,Exceptions & Corner Cases,"In the client credentials flow, should the access token response contain a refresh token?",8,1,False,True
2,Keyword Trap (synonym / role),Does the token response for a client acting on its own behalf include a refresh token?,9,1,False,True
3,Negation & Constraints,Can a refreshed access token have a broader scope than the original one?,3,2,True,True
4,Other: Buried chunk,Which parameters are required in the authorization request?,5,3,False,True
5,Limitation: re-ranking demotes the answer,Must the token endpoint response include Cache-Control no-store?,1,4,True,False
6,Control: rule stated in two places,"In the implicit flow, can the access token response include a refresh token?",3,2,True,True


## 8. Latency Profiling

Models are already loaded and warmed up above, so we time only the per-query work.
Each query is run `REPEATS` times and we report the mean and median.

* **Baseline retrieval** = encode the query with the bi-encoder + FAISS search over all chunks.
* **Re-ranking** = cross-encoder on the 10 candidate pairs + sort.
* **LLM generation** is timed separately. It is a network call, identical in both pipelines, and dominates end-to-end time.

In [24]:
REPEATS = 20
queries = [t["question"] for t in STRESS_TESTS]
timings = {"baseline": [], "rerank": []}

for q in queries:
    for _ in range(REPEATS):
        r = run_pipelines(q, generate=False)
        timings["baseline"].append(r["t_baseline_ms"])
        timings["rerank"].append(r["t_rerank_ms"])

def stats(xs):
    return {"mean_ms": statistics.mean(xs), "median_ms": statistics.median(xs), "p95_ms": float(np.percentile(xs, 95))}

base, rr = stats(timings["baseline"]), stats(timings["rerank"])
total2 = [a + b for a, b in zip(timings["baseline"], timings["rerank"])]
latency = pd.DataFrame({
    "Pipeline 1: bi-encoder top-10 (retrieval only)": base,
    "Cross-encoder re-rank of 10 pairs": rr,
    "Pipeline 2: bi-encoder + re-rank (retrieval only)": stats(total2),
}).T.round(2)

# One timed LLM call per pipeline for context (skipped without an API key)
if client is not None:
    llm_times = []
    for q in queries[:3]:
        top = [c["chunk_id"] for c in baseline_retrieve(q)[:TOP_N]]
        t = time.perf_counter(); generate_answer(q, top); llm_times.append((time.perf_counter() - t) * 1000)
    latency.loc["Gemini answer generation (per call)"] = [statistics.mean(llm_times), statistics.median(llm_times), max(llm_times)]

latency.to_csv("results/latency.csv")
print(f"{len(queries)} queries x {REPEATS} repeats, {len(chunks)} chunks in the index, CPU only")
latency.round(2)

7 queries x 20 repeats, 132 chunks in the index, CPU only


,mean_ms,median_ms,p95_ms
Pipeline 1: bi-encoder top-10 (retrieval only),12.38,12.18,15.27
Cross-encoder re-rank of 10 pairs,628.77,670.03,774.20
Pipeline 2: bi-encoder + re-rank (retrieval only),641.15,680.93,787.40
Gemini answer generation (per call),1550.28,1377.18,2259.32


## 9. Scaling Experiment: Cross-Encoder over the Whole Corpus

To support the scaling question, we compare re-ranking 10 candidates with running the cross-encoder over **every** chunk for a single query,
then extrapolate the per-pair cost to larger collections. The bi-encoder side stays cheap because chunk embeddings are precomputed and the
search is a single matrix-vector product (or an approximate index like HNSW at scale).

In [25]:
q = queries[0]
all_pairs = [(q, t) for t in chunk_texts]

t = time.perf_counter(); cross_encoder.predict(all_pairs[:TOP_K]); t_10 = time.perf_counter() - t
t = time.perf_counter(); cross_encoder.predict(all_pairs);          t_all = time.perf_counter() - t
per_pair_ms = t_all / len(all_pairs) * 1000

t = time.perf_counter(); baseline_retrieve(q); t_bi = (time.perf_counter() - t) * 1000

scaling = pd.DataFrame([
    {"corpus size (chunks)": TOP_K,       "cross-encoder time": f"{t_10 * 1000:.1f} ms (measured)"},
    {"corpus size (chunks)": len(chunks), "cross-encoder time": f"{t_all * 1000:.1f} ms (measured)"},
    *[{"corpus size (chunks)": n, "cross-encoder time": f"~{n * per_pair_ms / 1000:,.0f} s (extrapolated)"}
      for n in (10_000, 1_000_000, 100_000_000)],
])
print(f"Cross-encoder cost ≈ {per_pair_ms:.2f} ms per (query, chunk) pair; bi-encoder retrieval over the full index: {t_bi:.2f} ms")
scaling

Cross-encoder cost ≈ 45.89 ms per (query, chunk) pair; bi-encoder retrieval over the full index: 11.47 ms


,corpus size (chunks),cross-encoder time
0,10,503.3 ms (measured)
1,132,6057.5 ms (measured)
2,10000,~459 s (extrapolated)
3,1000000,"~45,890 s (extrapolated)"
4,100000000,"~4,589,030 s (extrapolated)"


## 10. Observations

### What happened in the rank tables
- In **4 out of 7** test questions (Q1, Q2, Q3, Q5), the correct chunk was **not** in the baseline top 3 (it was at rank #4 to #9).
- In all 4, the cross-encoder moved it into the top 3, and in 3 of them straight to **rank #1**. On average the correct chunk moved from **rank 6.5 to rank 1.5**.
- In Q4 and Q7 the baseline already had the right chunk at #3, and re-ranking moved it up to #2.
- Re-ranking is **not always better**. In Q6 the bi-encoder had the right chunk at #1, and the cross-encoder pushed it down to #4.

| Q | Question type | Correct chunk: baseline → re-ranked |
|---|---|---|
| Q1 | Keyword trap | Section 10.3 #4 → **#1** |
| Q2 | Exception | Section 4.4.3 #8 → **#1** |
| Q3 | Keyword trap (synonym) | Section 4.4.3 #9 → **#1** |
| Q4 | Negation ("MUST NOT") | Section 6 #3 → #2 |
| Q5 | Buried chunk | Section 4.1.1 #5 → **#3** |
| Q6 | Limitation | Section 5.1 #1 → #4 (got worse) |
| Q7 | Control | Section 4.2 #3 → #2 (both fine) |

### Why the bi-encoder (Pipeline 1) got it wrong
- The bi-encoder picks chunks that use the **same words** as the question, not the chunk that holds the **rule** being asked about.
- **Refresh-token questions (Q2, Q3):** the phrase "refresh token" pulled up the general Section 1.5 *Refresh Token* chunks to ranks #1–#3. The real answer is in Section 4.4.3: for the client credentials grant "A refresh token SHOULD NOT be included". That chunk was only #8 and #9.
- **Q1:** "access tokens" matched the definition in Section 1.4 (*Access Token*), which says nothing about keeping them secret. The actual rule, "MUST be kept confidential in transit and storage", is in the Security Considerations (Section 10.3).
- **Q3 (synonym):** I asked about "a client acting on its own behalf" instead of "client credentials grant". The bi-encoder could not connect the two (#9). The cross-encoder put Section 4.4.3 at #1.
- **Negation itself was not the main problem.** When I asked direct "MUST NOT" questions using the spec's own words (Q4, and several others I tried), the bi-encoder usually found the rule. It failed when the question's **topic words** pointed to a more general section.

### Answer 1 vs Answer 2
- **Answer 1 (baseline) was correct 3 times out of 7.** **Answer 2 (re-ranked) was correct 6 times out of 7.**
- **Omitted rule (Q2, Q3):** the baseline said "the provided context does not answer this", even though Section 4.4.3 answers it clearly.
- **Misstated rule (Q5):** this was the worst baseline answer. It said the required parameters are `response_type` "**token**" and `client_id`, which are the rules for the *implicit* grant only (Section 4.2.1), presented as the general answer. Answer 2 correctly said it depends on the grant type and gave `response_type=code` for the authorization code grant (Section 4.1.1).
- **Incomplete (Q1):** the baseline only said tokens must not be sent in the clear (Section 10.8). It missed "storage" and who may see the token. Answer 2 had Section 10.3 and gave the full rule.
- **No hallucination:** neither pipeline invented facts, because the prompt says to answer only from the context. The same LLM and prompt were used in both pipelines, so **the differences come only from retrieval**.

### Where re-ranking did not help
- **Q6 (re-ranking made it worse):** for "Must the token endpoint response include Cache-Control no-store?", the bi-encoder found the rule (Section 5.1) at #1. The cross-encoder preferred the example responses in Sections 4.4.3, 4.3.3 and 4.1.4, because they literally contain the text `Cache-Control: no-store`. So Answer 2 became unsure ("does not explicitly state that it *must*"), while Answer 1 was correct. The cross-encoder was trained on web search data (MS MARCO), so it can be fooled by text that *looks* like the answer.
- **Q7 (control):** the rule "no refresh token in the implicit grant" is written in two places (Sections 4.2 and 4.2.2). Section 4.2 was in the top 3 for both pipelines, so both answers were correct.
- Possible fixes: give the LLM more than 3 chunks, combine BM25 keyword search with dense search, or use a re-ranker fine-tuned on technical text.

### Latency
| Step | Average time |
|---|---|
| Pipeline 1: bi-encoder retrieval | ~12.4 ms |
| Cross-encoder re-ranking (10 chunks) | ~629 ms |
| Pipeline 2: total retrieval | ~641 ms |
| Gemini answer (median) | ~1.4 s |

- Re-ranking makes retrieval about **52 times slower** (12.4 ms → 641 ms).
- Gemini Flash-Lite was fast in this run (~1.4 s), so re-ranking adds about **45%** to the total response time (≈1.4 s → ≈2.0 s). That is a real cost, though still well under a second.
- For that extra ~0.6 s, correct answers went from 3/7 to 6/7. For a specification, where one "SHOULD NOT" changes the answer, I think it is worth it. On a GPU the re-ranking step would be much faster.

### Why not use the cross-encoder on all chunks?
- The cross-encoder must read the question and each chunk **together**, so nothing can be computed in advance. It needs one model run per chunk, for every question.
- In this run it cost about **46 ms per chunk**. Scoring all 132 chunks took ~6.1 s for one question, compared with ~11 ms for the bi-encoder.
- **1 million chunks would take about 12.7 hours per question**, and 100 million about 53 days.
- The bi-encoder's chunk vectors are computed once and stored in FAISS, so search stays fast. That is why we use **two stages**: the fast bi-encoder picks the top 10, and the slow, accurate cross-encoder re-ranks only those 10.

### Summary
On RFC 6749, re-ranking fixed the baseline's mistakes on exceptions, synonyms and keyword traps (Answer 2 correct in 6/7 vs 3/7). It is not perfect: it can be fooled by example text that only *looks* relevant (Q6), and it can only reorder what the bi-encoder already retrieved.
